# Toy BNN — paper figures

Loads per-split `.pt` files written by `sazz.scripts.uci_bnn` and produces
1. A metrics table for one (dataset, split) — for quick inspection.
2. The single-dataset 2×N predictive panel — for inspection.
3. **The paper figure**: a 4×5 grid of predictive bands across all four toy datasets and all five samplers (NUTS + four PDMPs). This is the figure that goes in the toy-BNN subsection.
4. **Appendix figure**: the matching 4×5 grid of epistemic-std-vs-x panels.

## Setup

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.scripts.uci_bnn import build_target
from sazz.models.bnn_torch import predict_regression

torch.set_default_dtype(torch.float64)

# ---- Pick what to inspect ----
DATASET     = "gap"   # boston / energy / naval / hernandez / gap / sharp / multiscale
SPLIT_ID    = 0
RESULTS_DIR = Path("results/uci_bnn")
TOY_DIR     = Path("datasets/toy_1d")

split_dir = RESULTS_DIR / DATASET / f"split_{SPLIT_ID:02d}"
print(f"Looking in {split_dir}")
print(f"  found: {sorted(p.name for p in split_dir.glob('*.pt'))}")


## Load all sampler runs

Each `.pt` is a self-contained payload: thinned samples, x_ref, layer_sizes,
metrics, etc. We load them into a dict keyed by sampler name.

In [ ]:
def load_runs(split_dir: Path) -> dict[str, dict]:
    runs = {}
    for pt_path in sorted(split_dir.glob("*.pt")):
        name = pt_path.stem
        runs[name] = torch.load(pt_path, weights_only=False)
    return runs

runs = load_runs(split_dir)
print(f"Loaded {len(runs)} samplers: {list(runs)}")

# Quick peek at one payload
example_sampler = list(runs)[0]
print(f"\nKeys in {example_sampler}.pt: {list(runs[example_sampler])}")


## Metrics summary

In [ ]:
import math

@torch.no_grad()
def predictive_summary(samples, target, X_test):
    """Posterior-predictive mean and std at each test point, on the
    standardised scale. Same as the runner's predict_regression but local
    so we can use it for both metrics and plotting."""
    likelihood = target.meta["model"].likelihood
    X_test = X_test.to(dtype=likelihood.X.dtype, device=likelihood.X.device)
    preds = torch.stack([
        likelihood.predict(beta, X_test).squeeze(-1) for beta in samples
    ])  # [n_samples, n_test]
    return preds.mean(0), preds.std(0)


def gaussian_log_lik(y_true, mean, pred_std, noise_std):
    total_std = (pred_std ** 2 + noise_std ** 2).sqrt()
    return (
        -0.5 * ((y_true - mean) / total_std) ** 2
        - total_std.log()
        - 0.5 * math.log(2 * math.pi)
    ).mean()


def ess_per_coord(samples, max_lag=None):
    x = samples - samples.mean(0, keepdim=True)
    n, d = x.shape
    var = (x ** 2).mean(0)
    if max_lag is None:
        max_lag = min(n - 1, 1000)
    rho_sum = torch.zeros(d, dtype=samples.dtype)
    prev_pair = torch.full((d,), float("inf"), dtype=samples.dtype)
    active = torch.ones(d, dtype=torch.bool)
    k = 1
    while k + 1 <= max_lag:
        c_k   = (x[:n - k]     * x[k:]).mean(0)     / var.clamp(min=1e-30)
        c_kp1 = (x[:n - k - 1] * x[k + 1:]).mean(0) / var.clamp(min=1e-30)
        pair = c_k + c_kp1
        kill = active & ((pair <= 0) | (pair >= prev_pair))
        active = active & ~kill
        rho_sum = rho_sum + torch.where(active, pair, torch.zeros_like(pair))
        prev_pair = torch.where(active, pair, prev_pair)
        if not active.any():
            break
        k += 2
    tau = 1.0 + 2.0 * rho_sum
    return n / tau.clamp(min=1.0)


def compute_metrics(samples, target, data, noise_std):
    """Recompute all metrics from the 4k saved samples, not the runner output."""
    mean_pred, std_pred = predictive_summary(samples, target, data["X_test"])
    rmse_std = ((mean_pred - data["y_test"]) ** 2).mean().sqrt()
    log_lik  = gaussian_log_lik(data["y_test"], mean_pred, std_pred, noise_std)
    ess      = ess_per_coord(samples)
    return {
        "rmse_std":      float(rmse_std),
        "rmse_orig":     float(rmse_std) * data["y_std"],
        "log_lik":       float(log_lik),
        "nll_orig":      -float(log_lik) + math.log(data["y_std"]),
        "pred_std_mean": float(std_pred.mean()),
        "ess_min":       float(ess.min()),
        "ess_median":    float(ess.median()),
        "ess_mean":      float(ess.mean()),
    }
    
def rebuild_target_for_predictions(payload):
    """Rebuild the BNNTarget that produced these samples so we can call
    predict_regression on them. Uses architecture/activation/noise stored
    in the .pt payload to reconstruct the same model on the same X_train."""
    # Reconstruct minimal config; build_target needs X_train, y_train.
    # For toys the data dict comes from datasets/toy_1d/<dataset>.pt.
    is_toy = (TOY_DIR / f"{DATASET}.pt").exists()
    if not is_toy:
        raise RuntimeError(
            f"Predictive plot requires a 1D toy dataset; {DATASET} isn't one."
        )
    data = torch.load(TOY_DIR / f"{DATASET}.pt", weights_only=False)

    # Build a fresh BNNConfig matching the saved one (in case the global
    # configs_for changed since you ran the experiment).
    from sazz.scripts.uci_bnn import BNNConfig
    cfg = BNNConfig(
        layer_sizes=payload["layer_sizes"],
        activation=payload["activation"],
        noise_std=payload["noise_std"],
        # The other fields don't affect predict_regression; defaults are fine.
    )
    target = build_target(data, cfg)
    return target, data, cfg




# Need a target to compute predictive metrics. Rebuild it once from any
# payload (they all share the same dataset and architecture for a given
# (dataset, split)).
first_payload = next(iter(runs.values()))
target, data, cfg = rebuild_target_for_predictions(first_payload)

rows = []
for name, payload in runs.items():
    samples = payload["samples"]
    if name == "map":
        # MAP is a single point estimate, not a posterior — skip the chain
        # metrics and only compute predictive ones.
        m = compute_metrics(samples, target, data, cfg.noise_std)
        m["ess_min"] = m["ess_median"] = m["ess_mean"] = float("nan")
    else:
        m = compute_metrics(samples, target, data, cfg.noise_std)
    rows.append({
        "sampler":          name,
        "n_samples":        samples.shape[0],
        "elapsed_sec":      payload.get("elapsed_sec"),
        **m,
    })

df = pd.DataFrame(rows).set_index("sampler")
df.round(4)

## Predictive bands, all toys × all samplers

Layout: rows = datasets, columns = samplers. NUTS is placed first as the reference. Within each row the y-limits are shared so bands are visually comparable across samplers. The plot shows the predictive mean and the ±2σ (≈95%) band combining epistemic and observation noise, on the original (un-standardised) scale.

We use a single representative split (the same `SPLIT_ID` set above) for every dataset. The story we want to tell is *posterior agreement*, not split variance, so showing one split is the right move; a multi-split version belongs in the appendix if anywhere.

In [ ]:
# ---------------------------------------------------------------------------
# Paper figure: 4 datasets x 5 samplers, predictive band only.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path

# Datasets and sampler order for the paper figure. Sampler order matters:
# NUTS is the reference and goes first; the four PDMPs follow in a fixed order.
PAPER_DATASETS = ["hernandez", "gap", "sharp", "multiscale"]
PAPER_SAMPLER_ORDER = [
    "nuts",
    "zigzag",
    "boomerang",
    "sticky_zigzag",
    "sticky_boomerang",
]

# Pretty labels for column titles and row labels
SAMPLER_LABELS = {
    "nuts":             "NUTS",
    "zigzag":           "Zig-Zag",
    "boomerang":        "Boomerang",
    "sticky_zigzag":    "Sticky Zig-Zag",
    "sticky_boomerang": "Sticky Boomerang",
}
DATASET_LABELS = {
    "hernandez":  "Hernandez",
    "gap":        "Gap",
    "sharp":      "Sharp",
    "multiscale": "Multiscale",
}

# Per-dataset axis limits — shared across the row so bands are comparable.
# (Reusing the same limits as the inspection cell above; tweak if needed.)
PAPER_LIMITS = {
    "hernandez":  {"xlim": (-6, 5), "ylim": (-75, 75)},
    "gap":        {"xlim": (-4, 4), "ylim": (-2.5, 3)},
    "sharp":      {"xlim": (-4, 4), "ylim": (-1, 2.5)},
    "multiscale": {"xlim": (-4, 4), "ylim": (-2, 2)},
}

SAMPLER_COLOURS = {
    "nuts":             "#1f77b4",  # blue
    "boomerang":        "#ff7f0e",  # orange
    "sticky_boomerang": "#d62728",  # red
    "zigzag":           "#2ca02c",  # green
    "sticky_zigzag":    "#9467bd",  # purple
    "map":              "#7f7f7f",  # grey
}

def load_dataset_runs(dataset: str, split_id: int):
    """Load all sampler payloads for one (dataset, split)."""
    split_dir = RESULTS_DIR / dataset / f"split_{split_id:02d}"
    runs = {}
    for pt_path in sorted(split_dir.glob("*.pt")):
        runs[pt_path.stem] = torch.load(pt_path, weights_only=False)
    return runs


def predictive_on_original_scale(payload, target, data, noise_std_std):
    """Posterior-predictive mean and total std on the *original* (un-standardised)
    output scale. Returns (mean_orig, epistemic_std_orig, total_std_orig)."""
    samples = payload["samples"]
    mean_s, epi_s = predictive_summary(samples, target, data["X_test"])
    y_std = data["y_std"]
    y_mean = data["y_mean"]
    mean_orig = mean_s.numpy() * y_std + y_mean
    epistemic_orig = epi_s.numpy() * y_std
    noise_orig = noise_std_std * y_std
    total_orig = np.sqrt(epistemic_orig ** 2 + noise_orig ** 2)
    return mean_orig, epistemic_orig, total_orig, noise_orig


def build_paper_figure(split_id: int, kind: str = "predictive"):
    """Build the 4xN_samplers paper figure.

    kind="predictive": mean + ±2σ band, true function, training points
    kind="epistemic":  epistemic std vs x (appendix figure)
    """
    n_rows = len(PAPER_DATASETS)
    n_cols = len(PAPER_SAMPLER_ORDER)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(2.4 * n_cols, 1.9 * n_rows),
        squeeze=False,
        sharex="row",
        sharey="row",
    )

    # We collect handles for a single shared legend at the bottom
    legend_handles, legend_labels = [], []

    for r, dataset in enumerate(PAPER_DATASETS):
        runs = load_dataset_runs(dataset, split_id)

        # Build the target/data once per dataset
        data_path = TOY_DIR / f"{dataset}.pt"
        data = torch.load(data_path, weights_only=False)
        from sazz.scripts.uci_bnn import BNNConfig, build_target
        sample_payload = next(iter(runs.values()))
        cfg = BNNConfig(
            layer_sizes=sample_payload["layer_sizes"],
            activation=sample_payload["activation"],
            noise_std=sample_payload["noise_std"],
        )
        target = build_target(data, cfg)

        x_test_raw = data["x_test_raw"]
        y_test_clean_raw = data.get("y_test_clean_raw", data["y_test_raw"])
        x_train_raw = data["x_train_raw"]
        y_train_raw = data["y_train_raw"]
        limits = PAPER_LIMITS.get(dataset, {})

        for c, sampler in enumerate(PAPER_SAMPLER_ORDER):
            ax = axes[r, c]

            if sampler not in runs:
                ax.text(0.5, 0.5, "missing",
                        ha="center", va="center",
                        transform=ax.transAxes,
                        fontsize=8, color="0.5")
                ax.set_xticks([]); ax.set_yticks([])
                continue

            payload = runs[sampler]
            mean_o, epi_o, total_o, noise_o = predictive_on_original_scale(
                payload, target, data, cfg.noise_std
            )
            colour = SAMPLER_COLOURS.get(sampler, f"C{c}")

            if kind == "predictive":
                # Order matters for visual layering
                h_band = ax.fill_between(
                    x_test_raw,
                    mean_o - 2 * total_o, mean_o + 2 * total_o,
                    color=colour, alpha=0.25, linewidth=0,
                )
                h_truth, = ax.plot(
                    x_test_raw, y_test_clean_raw,
                    color="k", lw=0.9, alpha=0.7,
                )
                h_mean, = ax.plot(
                    x_test_raw, mean_o,
                    color=colour, lw=1.4,
                )
                h_train = ax.scatter(
                    x_train_raw, y_train_raw,
                    s=8, facecolor="none", edgecolor="0.2",
                    linewidth=0.6,
                )
                if r == 0 and c == 0:
                    legend_handles = [h_truth, h_train, h_mean, h_band]
                    legend_labels = [
                        "true function",
                        "training data",
                        "posterior mean",
                        r"predictive $\pm 2\sigma$",
                    ]

            elif kind == "epistemic":
                ax.plot(x_test_raw, epi_o, color=colour, lw=1.4,
                        label="epistemic std")
                ax.axhline(noise_o, color="0.4", linestyle="--", lw=0.8,
                           label="noise std")
                ax.scatter(x_train_raw, np.zeros_like(x_train_raw),
                           s=15, color="0.4", marker="|", linewidth=0.8)
                if r == 0 and c == 0:
                    handles, labels_ = ax.get_legend_handles_labels()
                    legend_handles, legend_labels = handles, labels_
            else:
                raise ValueError(f"unknown kind={kind!r}")

            # Axis cosmetics
            if "xlim" in limits:
                ax.set_xlim(*limits["xlim"])
            if kind == "predictive" and "ylim" in limits:
                ax.set_ylim(*limits["ylim"])
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.15, linewidth=0.5)
            for spine in ("top", "right"):
                ax.spines[spine].set_visible(False)

            if r == 0:
                ax.set_title(SAMPLER_LABELS.get(sampler, sampler), fontsize=9)
            if c == 0:
                ax.set_ylabel(DATASET_LABELS.get(dataset, dataset),
                              fontsize=9, rotation=90, labelpad=8)
            if r == n_rows - 1:
                ax.set_xlabel("x", fontsize=8)

    # One shared legend at the bottom
    if legend_handles:
        fig.legend(
            legend_handles, legend_labels,
            loc="lower center", ncol=len(legend_handles),
            frameon=False, fontsize=8,
            bbox_to_anchor=(0.5, -0.02),
        )

    fig.tight_layout(rect=(0, 0.02, 1, 1))
    return fig


fig_main = build_paper_figure(SPLIT_ID, kind="predictive")
plt.show()


## Appendix figure: epistemic std vs x

Same 4×5 layout, but each panel shows the per-x epistemic standard deviation. The band should dip near training inputs (tick marks on the x-axis) and grow in extrapolation regions. This is the diagnostic that PDMPs are not just matching NUTS in the mean but also in the *shape* of uncertainty as a function of input.

In [ ]:
fig_app = build_paper_figure(SPLIT_ID, kind="epistemic")
plt.show()


## Save figures for the paper

Writes both figures to `figures/` as PDF (vector) and PNG (preview).

In [ ]:
# FIG_DIR = Path("figures")
# FIG_DIR.mkdir(exist_ok=True)

# for kind, stem in [("predictive", "toy_bnn_predictive"),
#                    ("epistemic",  "toy_bnn_epistemic")]:
#     fig = build_paper_figure(SPLIT_ID, kind=kind)
#     for ext in ("pdf", "png"):
#         out = FIG_DIR / f"{stem}.{ext}"
#         fig.savefig(out, dpi=300, bbox_inches="tight")
#         print(f"wrote {out}")
#     plt.close(fig)
